# NB04 — R², Residuals, and Sum-of-Squares Decomposition

> **Understanding what R² really measures — not just a formula.**


## 1. The three sums of squares

```
SS_tot = Σ(yᵢ − ȳ)²        Total variance in y
SS_res = Σ(yᵢ − ŷᵢ)²       Unexplained variance (residuals)
SS_reg = Σ(ŷᵢ − ȳ)²        Explained variance (regression)

SS_tot = SS_reg + SS_res     ← always true

R² = SS_reg / SS_tot  =  1 − SS_res / SS_tot
```

R² = 0.80 means "the model explains 80% of the variance in y".


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(0)
X = np.linspace(1, 10, 30)
y = 3 * X + np.random.normal(0, 4, 30) + 5

model = LinearRegression().fit(X.reshape(-1,1), y)
y_hat = model.predict(X.reshape(-1,1))
y_bar = y.mean()

ss_tot = np.sum((y - y_bar) ** 2)
ss_res = np.sum((y - y_hat) ** 2)
ss_reg = np.sum((y_hat - y_bar) ** 2)
r2     = 1 - ss_res / ss_tot

print(f"SS_tot = {ss_tot:.2f}")
print(f"SS_reg = {ss_reg:.2f}   (explained)")
print(f"SS_res = {ss_res:.2f}   (unexplained)")
print(f"SS_tot = SS_reg + SS_res: {np.isclose(ss_tot, ss_reg + ss_res)}")
print(f"R²     = {r2:.4f}  → model explains {r2*100:.1f}% of variance")


In [ ]:
# Visualise the decomposition for one data point
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'tot':'#2196F3','res':'#F44336','reg':'#4CAF50'}

for ax, (ss_name, y_ref, color, title) in zip(axes, [
    ('SS_tot', y_bar,  colors['tot'], 'SS_tot = Σ(y − ȳ)²
(total spread around mean)'),
    ('SS_res', y_hat,  colors['res'], 'SS_res = Σ(y − ŷ)²
(residuals around fitted line)'),
    ('SS_reg', y_bar,  colors['reg'], 'SS_reg = Σ(ŷ − ȳ)²
(fitted line spread around mean)'),
]):
    ax.scatter(X, y, color='steelblue', s=30, zorder=3, label='data')
    ax.plot(X, y_hat, 'k-', linewidth=1.5, label='fitted')
    ax.axhline(y_bar, color='gray', linestyle='--', linewidth=1, label='ȳ mean')
    ref = y_ref if hasattr(y_ref, '__len__') else np.full_like(X, y_ref)
    for xi, yi, ri in zip(X, y, ref):
        ax.vlines(xi, min(yi, ri), max(yi, ri), color=color, linewidth=2, alpha=0.6)
    ax.set_title(title); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Show how R² changes with noise level
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(1)
X_base = np.linspace(1, 10, 50)
noise_levels = [1, 4, 8, 15]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, noise in zip(axes, noise_levels):
    y = 3 * X_base + noise * np.random.randn(50) + 5
    m = LinearRegression().fit(X_base.reshape(-1,1), y)
    r2 = m.score(X_base.reshape(-1,1), y)
    ax.scatter(X_base, y, s=20, alpha=0.6, color='steelblue')
    ax.plot(X_base, m.predict(X_base.reshape(-1,1)), 'r-', linewidth=2)
    ax.set_title(f'noise={noise}\nR²={r2:.3f}')
    ax.grid(alpha=0.3)

plt.suptitle('R² decreases as noise increases', fontsize=12)
plt.tight_layout(); plt.show()


## 2. Limitations of R²

| Situation | R² misleads because... |
|-----------|----------------------|
| Adding irrelevant predictors | R² always increases even if predictors add no real information |
| Non-linear data | High R² on a linear fit doesn't mean the linear model is correct |
| Outliers | One outlier can dramatically change R² |
| No intercept | R² formula breaks down |

**Adjusted R²** corrects for the number of predictors:

```
R²_adj = 1 − (1−R²)(n−1)/(n−k−1)
```

where k = number of predictors, n = sample size.


In [ ]:
# Demonstrate R²_adj penalises irrelevant features
import numpy as np
from sklearn.linear_model import LinearRegression

np.random.seed(42)
n = 50
y = np.random.randn(n)    # y has NO real predictors

r2s, r2adjs = [], []
for k in range(1, 20):
    X_noise = np.random.randn(n, k)   # pure noise predictors
    m = LinearRegression().fit(X_noise, y)
    yh = m.predict(X_noise)
    ss_res = np.sum((y - yh)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res / ss_tot
    r2_adj = 1 - (1-r2)*(n-1)/(n-k-1)
    r2s.append(r2); r2adjs.append(r2_adj)

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(range(1,20), r2s,    'o-', label='R²', color='crimson')
plt.plot(range(1,20), r2adjs, 's--', label='Adjusted R²', color='steelblue')
plt.axhline(0, color='gray', linewidth=0.8)
plt.xlabel('Number of noise predictors'); plt.ylabel('Score')
plt.title('R² rises with irrelevant predictors; Adjusted R² stays near 0')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("With pure noise X, true R² should be ~0. R² inflates; Adjusted R² doesn't.")


## Key Takeaways

| Metric | Formula | When it's high |
|--------|---------|----------------|
| SS_tot | Σ(y−ȳ)² | y has lots of variance |
| SS_res | Σ(y−ŷ)² | model predicts poorly |
| SS_reg | Σ(ŷ−ȳ)² | model captures the variance |
| R² | 1 − SS_res/SS_tot | model explains most of the variance |
| Adjusted R² | 1 − (1−R²)(n−1)/(n−k−1) | even with multiple predictors |

**Next:** NB05 — t-statistics, p-values, and confidence intervals.
